In [1]:
import GEOparse, pandas as pd, numpy as np
from pathlib import Path

gse2 = GEOparse.get_GEO(geo="GSE43292", destdir="../data/bronze")

23-Nov-2026 13:25:09 DEBUG utils - Directory ../data/bronze already exists. Skipping.
23-Nov-2026 13:25:09 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE43nnn/GSE43292/soft/GSE43292_family.soft.gz to ../data/bronze\GSE43292_family.soft.gz
100%|██████████| 25.6M/25.6M [00:10<00:00, 2.55MB/s]  
23-Nov-2026 13:25:27 DEBUG downloader - Size validation passed
23-Nov-2026 13:25:27 DEBUG downloader - Moving C:\Users\WINDOW~1\AppData\Local\Temp\tmp_o7l77ei to C:\Users\Windows11\Documents\data\bronze\GSE43292_family.soft.gz
23-Nov-2026 13:25:27 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE43nnn/GSE43292/soft/GSE43292_family.soft.gz
23-Nov-2026 13:25:27 INFO GEOparse - Parsing ../data/bronze\GSE43292_family.soft.gz: 
23-Nov-2026 13:25:27 DEBUG GEOparse - DATABASE: GeoMiame
23-Nov-2026 13:25:27 DEBUG GEOparse - SERIES: GSE43292
23-Nov-2026 13:25:27 DEBUG GEOparse - PLATFORM: GPL6244
23-Nov-2026 13:25:29 DEBUG GEOparse - SAMPLE: GSM1060

In [2]:
print("Plateformes (GPL):", list(gse2.gpls.keys()))
expr2 = gse2.pivot_samples("VALUE")
pheno2 = gse2.phenotype_data
print("expr shape:", expr2.shape)
print("pheno shape:", pheno2.shape)
print(pheno2.columns.tolist())

Plateformes (GPL): ['GPL6244']
expr shape: (33297, 64)
pheno shape: (64, 33)
['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1.0.patient', 'characteristics_ch1.1.disease state', 'characteristics_ch1.2.tissue', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_laboratory', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count']


In [3]:
pheno2.iloc[0]

title                                           Patient 1, Macroscopically intact tissue
geo_accession                                                                 GSM1060117
status                                                             Public on Apr 01 2013
submission_date                                                              Jan 04 2013
last_update_date                                                             Apr 01 2013
type                                                                                 RNA
channel_count                                                                          1
source_name_ch1                        Macroscopically intact carotid tissue adjacent...
organism_ch1                                                                Homo sapiens
taxid_ch1                                                                           9606
characteristics_ch1.0.patient                                                          1
characteristics_ch1.1

In [4]:
for c in pheno2.columns:
    if c.startswith('characteristics') or c == 'source_name_ch1':
        print("\n==", c)
        print(pheno2[c].value_counts().head(10))


== source_name_ch1
source_name_ch1
Macroscopically intact carotid tissue adjacent to the atheroma plaque    32
Carotid atheroma plaque obtained from endarterectomy                     32
Name: count, dtype: int64

== characteristics_ch1.0.patient
characteristics_ch1.0.patient
1     2
2     2
3     2
4     2
5     2
6     2
7     2
8     2
9     2
10    2
Name: count, dtype: int64

== characteristics_ch1.1.disease state
characteristics_ch1.1.disease state
hypertension    64
Name: count, dtype: int64

== characteristics_ch1.2.tissue
characteristics_ch1.2.tissue
Macroscopically intact tissue    32
Atheroma plaque                  32
Name: count, dtype: int64


In [5]:
print("NaN:", int(expr2.isna().sum().sum()))
print(expr2.stack().describe())
gpl2 = gse2.gpls[list(gse2.gpls.keys())[0]].table
print(gpl2.shape)
print(gpl2.columns.tolist())
print(gpl2.iloc[2].to_string()[:1200])

NaN: 0
count    2.131008e+06
mean     6.195365e+00
std      2.233342e+00
min      1.706570e+00
25%      4.431120e+00
50%      6.146760e+00
75%      7.709610e+00
max      1.430770e+01
dtype: float64
(33297, 12)
['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND', 'RANGE_START', 'RANGE_STOP', 'total_probes', 'gene_assignment', 'mrna_assignment', 'category']
ID                                                           7896740
GB_LIST            NM_001004195,NM_001005240,NM_001005484,BC13684...
SPOT_ID                                             chr1:69091-70008
seqname                                                         chr1
RANGE_GB                                                NC_000001.10
RANGE_STRAND                                                       +
RANGE_START                                                    69091
RANGE_STOP                                                     70008
total_probes                                                      24
gene_a

In [6]:
t = pheno2['characteristics_ch1.2.tissue']
pheno2_clean = pd.DataFrame({
    'sample_id': pheno2['geo_accession'],
    'group': np.where(t == 'Atheroma plaque', 'disease', 'control'),
    'patient': pheno2['characteristics_ch1.0.patient'].astype(int),
    'site': 'carotid artery',
    'sex': np.nan,
    'age': np.nan,
    'platform': pheno2['platform_id'],
}).reset_index(drop=True)

print(pheno2_clean['group'].value_counts())
assert pheno2_clean.groupby('patient')['group'].nunique().eq(2).all()   # chaque patient a plaque + intact
expr2.index = expr2.index.astype(str)
assert set(expr2.columns) == set(pheno2_clean['sample_id'])
expr2 = expr2[pheno2_clean['sample_id']]

group
control    32
disease    32
Name: count, dtype: int64


In [7]:
def first_symbol(s):
    if not isinstance(s, str) or s.strip() in ('', '---'):
        return None
    parts = s.split(' /// ')[0].split(' // ')
    if len(parts) > 1 and parts[1].strip() not in ('', '---'):
        return parts[1].strip()
    return None

ann2 = gpl2[gpl2['category'] == 'main'].copy()
ann2['symbol'] = ann2['gene_assignment'].map(first_symbol)
ann2 = ann2.dropna(subset=['symbol'])
ann2['ID'] = ann2['ID'].astype(str)
mapping2 = ann2.drop_duplicates('ID').set_index('ID')['symbol']

e2 = expr2.loc[expr2.index.intersection(mapping2.index)].copy()
e2['gene'] = e2.index.map(mapping2)
expr2_gene = e2.groupby('gene').median()
expr2_gene.columns.name = None

print("probesets avant:", expr2.shape[0], "| annotés:", e2.shape[0],
      "| gènes uniques:", expr2_gene.shape[0])
print(expr2_gene.loc[expr2_gene.index.isin(['BTK', 'ITGAL'])].iloc[:, :3])

probesets avant: 33297 | annotés: 25293 | gènes uniques: 23307
       GSM1060117  GSM1060118  GSM1060119
gene                                     
BTK       7.89421     8.43597     7.93167
ITGAL     7.00099     7.48828     7.08405


In [8]:
out2 = Path('../data/silver/GSE43292')
out2.mkdir(parents=True, exist_ok=True)
expr2_gene.reset_index().to_parquet(out2 / 'expr.parquet')
pheno2_clean.to_parquet(out2 / 'pheno.parquet')

chk2 = pd.read_parquet(out2 / 'expr.parquet')
print(chk2.shape, chk2.columns[:3].tolist())

(23307, 65) ['gene', 'GSM1060117', 'GSM1060118']


In [9]:
a = set(pd.read_parquet('../data/silver/GSE100927/expr.parquet')['gene'])
b = set(chk2['gene'])
print("gènes communs AS discovery ∩ validation:", len(a & b))

gènes communs AS discovery ∩ validation: 12823
